In [ ]:
# Install dependencies
!pip install -U datasets huggingface_hub fsspec transformers

# Load dataset
from datasets import load_dataset
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1') #This is the data it is tuning on, this can be academic papers, business magazines etc

# Clean dataset
def clean_text(example):
    example['text'] = example['text'].strip().replace('\n', ' ')
    return example

dataset = dataset.map(clean_text)

# Load pretrained GPT-2 tokenizer and model
from transformers import AutoTokenizer, GPT2LMHeadModel

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# GPT-2 has no pad token by default
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2")

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Setup data collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # For causal language modeling
)

# Define training arguments
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./gpt2_finetuned_output",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_steps=500,
    logging_steps=100,
    report_to="none"  # Disable wandb
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"].select(range(10000)),
    tokenizer=tokenizer
)

# Fine-tune the model
trainer.train()

# Save the fine-tuned model and tokenizer
model.save_pretrained("gpt2_finetuned")
tokenizer.save_pretrained("gpt2_finetuned")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/733k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/6.36M [00:00<?, ?B/s]

...

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

/tmp/ipython-input-1-1185868273.py:57: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.950600
200,3.647300
300,3.678700
400,3.629700
500,3.628200
600,3.611500
700,3.507100
800,3.535200
900,3.484600
1000,3.545400


('gpt2_finetuned/tokenizer_config.json',
 'gpt2_finetuned/special_tokens_map.json',
 'gpt2_finetuned/vocab.json',
 'gpt2_finetuned/merges.txt',
 'gpt2_finetuned/added_tokens.json',
 'gpt2_finetuned/tokenizer.json')

In [ ]:

'''
Comparing the weights of original model and tuned model. This is standard fine tuning.
This is not LoRA, PEFT, or instruction tuning (prompt tuning).
In this model
- All weights are updated
- Tokenizer is reused from GPT-2
- Suitable for domain adaptation, continued pretraining, or experimentation
'''

# Load the original pretrained model (reference)
original_model = GPT2LMHeadModel.from_pretrained("gpt2")
# Load the fine-tuned model
finetuned_model = GPT2LMHeadModel.from_pretrained("gpt2_finetuned")

# Check how many parameters differ
total_params = 0
changed_params = 0

import torch
for name, param in finetuned_model.named_parameters():
    orig_param = dict(original_model.named_parameters())[name]
    total_params += param.numel()
    if not torch.allclose(param, orig_param, atol=1e-5):  # slight tolerance
        changed_params += param.numel()

print(f"Total parameters: {total_params}")
print(f"Changed parameters: {changed_params}")
print(f"Percentage changed: {changed_params / total_params * 100:.2f}%")


Total parameters: 124439808
Changed parameters: 124439808
Percentage changed: 100.00%


In [ ]:
from transformers import GPT2LMHeadModel, AutoTokenizer, pipeline

# Load original GPT-2
original_model = GPT2LMHeadModel.from_pretrained("gpt2")
original_tokenizer = AutoTokenizer.from_pretrained("gpt2")
original_tokenizer.pad_token = original_tokenizer.eos_token  # ensure compatibility
original_generator = pipeline("text-generation", model=original_model, tokenizer=original_tokenizer)

# Load fine-tuned GPT-2
ft_model = GPT2LMHeadModel.from_pretrained("gpt2_finetuned")
ft_tokenizer = AutoTokenizer.from_pretrained("gpt2_finetuned")
ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_generator = pipeline("text-generation", model=ft_model, tokenizer=ft_tokenizer)

# Same test prompt
prompt = "Artificial intelligence is"

# Generate from original model
original_output = original_generator(prompt, max_length=50, do_sample=True)[0]["generated_text"]

# Generate from fine-tuned model
ft_output = ft_generator(prompt, max_length=50, do_sample=True)[0]["generated_text"]

# Compare
print("----Original GPT-2 Output:\n", original_output)
print("\n-----Fine-Tuned GPT-2 Output:\n", ft_output)

'''
Original GPT-2 Output:
 Artificial intelligence is the world's top science or technology startup that has a massive investment in its research and development. This is true even when companies like Google or Facebook are working to bring artificial intelligence to your life. It's a big deal that artificial intelligence is being used to make your life easier for you.

 I've been a programmer since the age of 12. I was the first programmer to do anything like this. I learned to read and write code in a school environment and to create visual prototypes of my design to test the ideas before I showed it to other programmers.

 I have a real interest in helping people that are struggling. I've been working with people who are working on projects like this for a while now. I've had several friends who have had their work published in books and they've been doing great. It just so happens that the way I write code is more interesting to me.

 I've been interested in helping people. I've been working with people who are working on projects like this for a while now. I've had several friends who have had their work published in books and they've been doing great. It just so happens that the way I write code is more interesting to me. I've been working with people who are working on projects
----------------------------------

Fine-Tuned GPT-2 Output:
 Artificial intelligence is becoming a large part of the lives of people, especially in the developing world, and it is becoming increasingly difficult to predict what a person's future will be. There is a growing emphasis, and the growing popularity of artificial intelligence, on the virtues of self @-@ regulation, a concept that has been heavily promoted by the Industrial Revolution. The goal of this is to eliminate the need for manual labor, which can be a source of stress for many workers. The focus of artificial intelligence research is on the development of self @-@ regulation, and on the development of machine learning algorithms to solve complex problems. The emphasis is on the recognition that humans are capable of doing complex things, but that they need to be able to process and process them well, and that this requires the ability to empathize with other humans and empathize with each other. This emphasis is based partly on the fact that humans are not the only species to develop self @-@ regulation, but it also emphasizes the importance of human self @-@ mastery, which is a fundamental part of the human condition. There is also a growing interest in the problem of self @-@ regulation, which has been widely criticized, mainly for its reliance on artificial intelligence. The goal of artificial intelligence research is to
'''

Device set to use cuda:0
Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by pr

🔹 Original GPT-2 Output:
 Artificial intelligence is the world's top science or technology startup that has a massive investment in its research and development. This is true even when companies like Google or Facebook are working to bring artificial intelligence to your life. It's a big deal that artificial intelligence is being used to make your life easier for you.

I've been a programmer since the age of 12. I was the first programmer to do anything like this. I learned to read and write code in a school environment and to create visual prototypes of my design to test the ideas before I showed it to other programmers.

I have a real interest in helping people that are struggling. I've been working with people who are working on projects like this for a while now. I've had several friends who have had their work published in books and they've been doing great. It just so happens that the way I write code is more interesting to me.

I've been interested in helping people. I've been w